# Register a Local MLflow Model for the API

This notebook builds a small demo taxi-duration model, logs it to a local MLflow tracking server, and registers it for the FastAPI application. The goal is to keep the deployment workflow self-contained and runnable on your machine.

## Workflow

1. Create a compact demo dataset.
2. Train a simple scikit-learn regression pipeline.
3. Log the model to the local MLflow server.
4. Register the model and assign the `production` alias.

In [8]:
import logging
import os
from datetime import UTC, datetime
from pathlib import Path

import mlflow
import pandas as pd
from dotenv import load_dotenv
from mlflow import MlflowClient
from mlflow.exceptions import MlflowException
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

# Show only errors from mlflow.utils; skip info and warning messages.
logging.getLogger("mlflow.utils").setLevel(logging.ERROR)

In [9]:
PROJECT_ROOT = Path.cwd() if (Path.cwd() / ".env").exists() else Path.cwd().parent
load_dotenv(dotenv_path=PROJECT_ROOT / ".env")

MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", "http://127.0.0.1:5000")
REGISTERED_MODEL_NAME = os.getenv("REGISTERED_MODEL_NAME", "lr-ride-duration")
MODEL_ALIAS = os.getenv("MODEL_ALIAS", "production")

print(
    {
        "tracking_uri": MLFLOW_TRACKING_URI,
        "registered_model_name": REGISTERED_MODEL_NAME,
        "model_alias": MODEL_ALIAS,
    }
)

{'tracking_uri': 'http://127.0.0.1:5000', 'registered_model_name': 'lr-ride-duration', 'model_alias': 'production'}


## Build a Small Demo Dataset

The dataset is intentionally tiny. It is only here to make the deployment path runnable end to end, not to produce a production-quality model.

In [10]:
rides = pd.DataFrame(
    [
        {
            "PULocationID": 161,
            "DOLocationID": 236,
            "trip_distance": 1.2,
            "duration_minutes": 8,
        },
        {
            "PULocationID": 161,
            "DOLocationID": 236,
            "trip_distance": 2.8,
            "duration_minutes": 14,
        },
        {
            "PULocationID": 142,
            "DOLocationID": 239,
            "trip_distance": 1.7,
            "duration_minutes": 10,
        },
        {
            "PULocationID": 142,
            "DOLocationID": 239,
            "trip_distance": 4.1,
            "duration_minutes": 19,
        },
        {
            "PULocationID": 138,
            "DOLocationID": 170,
            "trip_distance": 3.3,
            "duration_minutes": 16,
        },
        {
            "PULocationID": 138,
            "DOLocationID": 170,
            "trip_distance": 5.6,
            "duration_minutes": 24,
        },
        {
            "PULocationID": 186,
            "DOLocationID": 230,
            "trip_distance": 2.1,
            "duration_minutes": 11,
        },
        {
            "PULocationID": 186,
            "DOLocationID": 230,
            "trip_distance": 6.2,
            "duration_minutes": 27,
        },
        {
            "PULocationID": 107,
            "DOLocationID": 79,
            "trip_distance": 1.5,
            "duration_minutes": 9,
        },
        {
            "PULocationID": 107,
            "DOLocationID": 79,
            "trip_distance": 4.8,
            "duration_minutes": 21,
        },
    ]
)
rides["trip_route"] = (
    rides["PULocationID"].astype(str) + "_" + rides["DOLocationID"].astype(str)
)
rides

,PULocationID,DOLocationID,trip_distance,duration_minutes,trip_route
0,161,236,1.2,8,161_236
1,161,236,2.8,14,161_236
2,142,239,1.7,10,142_239
3,142,239,4.1,19,142_239
4,138,170,3.3,16,138_170
5,138,170,5.6,24,138_170
6,186,230,2.1,11,186_230
7,186,230,6.2,27,186_230
8,107,79,1.5,9,107_79
9,107,79,4.8,21,107_79


## Train a Small Regression Pipeline

We keep the pipeline simple: one-hot encode the route, keep the distance as a numeric feature, and fit a linear regression model.

In [11]:
feature_columns = ["trip_route", "trip_distance"]
target_column = "duration_minutes"

X = rides[feature_columns]
y = rides[target_column]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

preprocessor = ColumnTransformer(
    transformers=[
        ("route", OneHotEncoder(handle_unknown="ignore"), ["trip_route"]),
    ],
    remainder="passthrough",
)

pipeline = Pipeline(
    [
        ("preprocessor", preprocessor),
        ("model", LinearRegression()),
    ]
)

pipeline.fit(X_train, y_train)
predictions = pipeline.predict(X_test)
mae = mean_absolute_error(y_test, predictions)

comparison = X_test.copy()
comparison["actual_duration"] = y_test.round(2)
comparison["predicted_duration"] = pd.Series(predictions, index=X_test.index).round(2)
comparison.sort_index()

,trip_route,trip_distance,actual_duration,predicted_duration
1,161_236,2.8,14,14.18
5,138_170,5.6,24,24.89
8,107_79,1.5,9,8.25


The table above gives a quick before-and-after check: the model is simple, but it is accurate enough for a local deployment demo. We will log the MAE to MLflow so the run keeps a lightweight evaluation record.

In [12]:
print({"mae_minutes": round(mae, 3)})

{'mae_minutes': 0.606}


## Log the Model to Local MLflow

Make sure the MLflow server from [01-run-local-mlflow.md](../01-run-local-mlflow.md) is running before you execute this cell.

In [13]:
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment("local-taxi-duration-demo")

with mlflow.start_run(run_name="linear-regression-demo") as run:
    mlflow.log_param("model_type", "LinearRegression")
    mlflow.log_param("feature_columns", ",".join(feature_columns))
    mlflow.log_metric("mae_minutes", mae)
    model_info = mlflow.sklearn.log_model(
        sk_model=pipeline,
        name="model",
        input_example=X_train.head(2),
    )
    run_id = run.info.run_id

print({"run_id": run_id, "tracking_uri": mlflow.get_tracking_uri()})

2026/09/30 06:03:58 INFO mlflow.tracking.fluent: Experiment with name 'local-taxi-duration-demo' does not exist. Creating a new experiment.


🏃 View run linear-regression-demo at: http://127.0.0.1:5000/#/experiments/1/runs/1cebc3ad7aa14ecb8502f9555f871e64
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
{'run_id': '1cebc3ad7aa14ecb8502f9555f871e64', 'tracking_uri': 'http://127.0.0.1:5000'}


## Register or Reuse the Model Name

If the registered model already exists, we reuse it and add a new version. That keeps the API pointed at a stable model name while you iterate.

In [14]:
client = MlflowClient()

try:
    client.create_registered_model(
        name=REGISTERED_MODEL_NAME,
        tags={"framework": "scikit-learn"},
        description="Demo model for the local FastAPI deployment workflow.",
    )
    print("Created registered model.")
except MlflowException as error:
    # Reuse the model only if it already exists; re-raise any other MLflow error.
    if error.error_code != "RESOURCE_ALREADY_EXISTS":
        raise
    print("Registered model already exists. Reusing it.")

model_version = client.create_model_version(
    name=REGISTERED_MODEL_NAME,
    source=model_info.model_uri,
    run_id=run_id,
)

print({"version": model_version.version})

2026/09/30 06:04:24 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: lr-ride-duration, version 1


Created registered model.
{'version': '1'}


## Promote the Version with an Alias

The API reads the model by alias instead of hard-coding a version number. That makes it easier to promote a later version without editing the service code.

In [15]:
client.set_registered_model_alias(
    name=REGISTERED_MODEL_NAME,
    alias=MODEL_ALIAS,
    version=model_version.version,
)

client.update_model_version(
    name=REGISTERED_MODEL_NAME,
    version=model_version.version,
    description=f"Promoted locally on {datetime.now(tz=UTC).date()} for the FastAPI demo.",
)

print({"alias": MODEL_ALIAS, "version": model_version.version})

{'alias': 'production', 'version': '1'}


## Inspect Registered Versions

This final table gives you a quick check that the alias is attached to the version you just created.

In [16]:
versions = client.search_model_versions(filter_string=f"name='{REGISTERED_MODEL_NAME}'")

version_table = pd.DataFrame(
    [
        {
            "version": version.version,
            "aliases": ", ".join(
                client.get_model_version(REGISTERED_MODEL_NAME, version.version).aliases
                or []
            ),
            "run_id": version.run_id,
        }
        for version in versions
    ]
)

version_table.sort_values("version", ascending=False).reset_index(drop=True)

,version,aliases,run_id
0,1,production,1cebc3ad7aa14ecb8502f9555f871e64


If the table shows your `production` alias, you can now move to [02-run-the-api-locally.md](../02-run-the-api-locally.md) and start the FastAPI service.